# Market Tags Walkthrough — Unlucky, Overlooked, Confidence Tiers & Box Comps

**Handoff notebook for [data team lead] / [program supervisor] — [data partner] internship, Project 2 (배당 패턴 분석)**

**v3 note:** the tag definitions below were tightened after finding that the same race could produce several simultaneous Unlucky/Overlooked tags at once (worst case: 5 Overlooked tags in one 14-horse race) -- see the new Section 3b for the full diagnosis and fix. Population numbers throughout this notebook reflect the tightened (v3) definition.

Written by 성지환 (Jihwan) so that someone with zero prior context on this code can read it top to bottom and understand: what these products are, why we believe the numbers, how the code works line by line, and what's still open/unfinished.

---

## What this notebook covers

Four things you asked me to hand off — **Unlucky tag, Overlooked tag ("overconfidence"), confidence tiers, and box comps** — turned out to live in **one single file**: `code/market_tags.py`. They're not four separate systems; they're four facets of the same module, because they share the same underlying data (the reference pools) and the same matching logic. This notebook walks through all four together, in the order you'd actually explain them to someone.

## One-paragraph plain-English summary

For every horse in a race, we have three independent opinions: what the **betting public / expert tipsters** think (`expert_score`), what our **statistical model** thinks (`fund_p`, a win-probability from the fundamental rating engine), and what **actually happened** (`placed`). Most of the time these roughly agree. Sometimes they disagree sharply and *in a specific, provable direction* — and when they do, that disagreement itself turns out to predict something real about the outcome. Two such patterns are proven enough to show customers. This notebook explains both, plus the machinery (box comps, confidence tiers) we built to present them honestly.

## Where this fits in the bigger picture

- **`fund_p`** (the "stats say" side) comes from the fundamental rating engine — the same family of model as **BB 지수 (bb_rating)**, which I already handed off separately (`bb_rating_walkthrough.ipynb`). It is **not** the exact same fitted model/vintage as bb_rating's production score — more on this in Section 9 (there's a known ~0.94 correlation, not an exact match, and that's expected/documented, not a bug).
- **`expert_score`** comes from a separate table of tipster picks (`expect_5horse` in the RDS database), aggregated and cleaned to exclude a few pseudo-experts per [data team lead]'s exclusion list (e.g. `배당누적`, `배당판인기도` — these are odds-derived, not real independent opinions, so including them would be circular).
- Everything here is downstream of the fundamental model, but this module itself doesn't refit anything — it just reads two CSV "reference pools" and does nearest-neighbor lookups. That's why it's lightweight and fast.


## 1. Vocabulary — the building blocks

These terms show up constantly below, so here they are up front in plain language.

| Term | What it means |
|---|---|
| `odds_final` | The final Korean pari-mutuel win odds (e.g. `5.5` means a ₩1,000 bet returns ₩5,500 if the horse wins). **Lower odds = the betting market thinks the horse is more likely to win.** |
| `expert_score` | An aggregated score built from tipster/expert picks (how many experts picked this horse, weighted). **Higher = experts favor it more.** Roughly ranges 0–130+. |
| `fund_p` | This horse's estimated win probability from the **fundamental statistical model** (Bolton & Chapman style, same family as bb_rating). This is the "what the data says, independent of public opinion" number. All `fund_p` values in one race sum to ~1. |
| `fund_p_race_rank` | `fund_p` converted to a **percentile rank within just this one race** (0 to 1, via `pandas.rank(pct=True)`). We use the rank instead of the raw probability because races have different field sizes — "top third of a 8-horse field" and "top third of a 14-horse field" need a common scale. |
| `log_odds` | `log(odds_final)`. Odds are heavily skewed (a 2.0 vs a 3.0 favorite is a big gap; a 80 vs 90 long shot is not), so taking the log makes the "distance" between two horses' odds much more meaningful for comparison. |
| `placed` | Whether the horse finished in the paying positions for 연승식 (place betting) — top 2 for small fields, top 3 for 8+ starters. `1` = yes, `0` = no. |
| `tercile` | Splitting a group into 3 equal-sized buckets (bottom third / middle third / top third). Used here to define "experts like it" vs "nobody picked it." |
| `z-score / standardizing` | Rescaling a number as `(value − mean) / std_dev`, so two different-scaled variables (like `log_odds` and `fund_p_race_rank`) can be compared fairly on the same footing before computing a distance between them. |
| `Euclidean distance` | The ordinary straight-line distance formula, `sqrt(Σ(a-b)²)`, applied here to the standardized `(log_odds, fund_p_race_rank)` pair — this is how we find the "most similar" historical horses. |
| `CI` (confidence interval) | A range that's likely to contain the true value. "CI excludes 1.0" is the key phrase throughout this notebook — it means we're confident the ratio is really different from 1 (i.e., a real effect), not just noise. |
| `ratio` | Actual place rate ÷ market-implied place rate (the implied rate comes from a Harville model on the final odds — i.e., "what should have happened if the market's odds were exactly right"). **Ratio < 1 means these horses underperformed what the market expected of them.** |


## 2. Setup

Load the module and point it at the data folder. In production `market_tags.py` expects its two reference-pool CSVs sitting next to it; here we override the paths to point at `data/` explicitly (this is exactly what the `TAG_CONFIG` dict is designed for — it's not a hack, it's the documented extension point).


In [ ]:
import sys, os
from pathlib import Path

# If auto-detection below can't find the project folder, just paste its full path
# here and re-run this cell. Example (Mac default location):
# PROJECT_ROOT_OVERRIDE = "/Users/jihwansung/Documents/market_tags_walkthrough"
PROJECT_ROOT_OVERRIDE = None

MARKER = "code/market_tags.py"

def _ok(base: Path) -> bool:
    try:
        return (base / MARKER).exists()
    except OSError:
        return False

def find_project_root():
    if PROJECT_ROOT_OVERRIDE:
        base = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if _ok(base):
            return base
        raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE is set to {base}, but {MARKER} isn't there.")

    # 1) VS Code's Jupyter extension exposes the actual .ipynb path in this global
    vsc_path = globals().get("__vsc_ipynb_file__")
    if vsc_path:
        base = Path(vsc_path).resolve().parent
        if _ok(base):
            return base

    # 2) Classic Jupyter Notebook / JupyterLab: ask the running server which file this is
    try:
        import ipynbname
        base = ipynbname.path().resolve().parent
        if _ok(base):
            return base
    except Exception:
        pass

    # 3) The folder's known default delivery location
    for known in (Path.home() / "Documents" / "market_tags_walkthrough",
                  Path.home() / "Desktop" / "market_tags_walkthrough"):
        if _ok(known):
            return known.resolve()

    # 4) Current working directory, a market_tags_walkthrough subfolder of it,
    #    or any of its parent directories
    cwd = Path.cwd().resolve()
    for base in [cwd, cwd / "market_tags_walkthrough", *cwd.parents]:
        if _ok(base):
            return base

    # 5) Last resort: bounded search under common folders, skipping system/junk dirs
    SKIP = {"Library", "Applications", "node_modules", ".git", "__pycache__", "venv", ".venv"}
    MAX_DEPTH = 6
    for root in [Path.home() / "Documents", Path.home() / "Desktop", Path.home()]:
        if not root.exists():
            continue
        for dirpath, dirnames, _ in os.walk(root):
            rel_parts = Path(dirpath).relative_to(root).parts
            if len(rel_parts) >= MAX_DEPTH:
                dirnames[:] = []
                continue
            dirnames[:] = [d for d in dirnames if d not in SKIP and not d.startswith(".")]
            if Path(dirpath).name == "market_tags_walkthrough" and _ok(Path(dirpath)):
                return Path(dirpath).resolve()

    raise FileNotFoundError(
        "Could not auto-locate the market_tags_walkthrough folder from any of the "
        "usual signals (VS Code, Jupyter server, default Documents/Desktop location, "
        "working directory, or a scan of your home folder). Set PROJECT_ROOT_OVERRIDE "
        "at the top of this cell to the folder's full path and re-run -- e.g. "
        "'/Users/jihwansung/Documents/market_tags_walkthrough'."
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)          # so every relative "data/..." path below just works
sys.path.append(str(PROJECT_ROOT / "code"))
print("Project root resolved to:", PROJECT_ROOT)

import market_tags as mt
import pandas as pd
import numpy as np

# Point the module at our data/ folder (see market_tags.py's TAG_CONFIG dict)
mt.TAG_CONFIG["unlucky"]["pool_path"] = "data/unlucky_reference_pool.csv"
mt.TAG_CONFIG["overlooked"]["pool_path"] = "data/overlooked_reference_pool.csv"

pd.set_option("display.max_columns", 20)
print("market_tags loaded OK. MIN_COMPS =", mt.MIN_COMPS, "| DEFAULT_K =", mt.DEFAULT_K)
print("Default tier cutpoints:", mt.DEFAULT_TIER_CUTPOINTS, "| tight_window:", mt.TIGHT_WINDOW)


## 3. The core idea — two kinds of disagreement

Both tags are defined by crossing **"what experts think"** with **"what the stats think"**, using fixed thresholds locked from the validated build (`market_tags.py` lines 55-58):

| Threshold | Value | Meaning |
|---|---|---|
| `EXPERT_HIGH_MIN` | `expert_score >= 97` | population-absolute — "experts like it" |
| `EXPERT_LOW_MAX` | `expert_score <= 23` | population-absolute — "nobody picked it" |
| `EXPERT_RACE_RANK_HIGH_MIN` | `expert_score_race_rank >= 0.667` | **(v3)** must ALSO be top third of *this race's* experts |
| `EXPERT_RACE_RANK_LOW_MAX` | `expert_score_race_rank <= 0.333` | **(v3)** must ALSO be bottom third of *this race's* experts |
| `MIN_EXPERT_COVERAGE` | `>= 3 horses with expert_score > 0` | **(v3)** race must have real tipster engagement to be tag-eligible at all |
| `FUND_RANK_WORSE_MAX` | `fund_p_race_rank <= 0.333` | bottom third of the field by stats |
| `FUND_RANK_BETTER_MIN` | `fund_p_race_rank >= 0.667` | top third of the field by stats |

The `EXPERT_RACE_RANK_*` and `MIN_EXPERT_COVERAGE` rows are new in v3 — see Section 3b for why they exist. Without them, `expert_score` was judged only against the whole population, never against the specific field it's racing in, and that mismatch is what let multiple horses in the same race qualify at once.

|  | stats say worse (bottom 1/3) | stats say better (top 1/3) |
|---|---|---|
| **experts like it** (top 1/3) | → **Unlucky** | (no tag — they agree) |
| **nobody picked it** (bottom 1/3) | (no tag — they agree) | → **Overlooked** |

If a horse doesn't land in either off-diagonal cell, it gets **no tag** — most horses, most of the time, since experts and stats usually roughly agree.

**Important — "overconfidence" naming note:** you mentioned wanting a walkthrough on "the overconfidence index." Confirmed with you this refers to the **Overlooked** tag above (not a separate thing, and not the same as "Benter's overconfidence correction" inside bb_rating, which is a different concept — see Section 9).


## 3b. One race, multiple tags — the problem we found and fixed (v3)

Before shipping, we checked something that isn't obvious from the definitions alone: **can the same race produce more than one Unlucky or Overlooked tag at once?** If a "surprise" tag fires on a third of the field, it stops meaning anything special about any one horse.

**Checked directly against the real historical join (5,878 races).** Under the original definition:

| | Races with ≥1 tag | Races with **>1** tag at once | Worst case seen |
|---|---|---|---|
| Unlucky | 462 | 16 (3.5%) | 2 horses in one race |
| Overlooked | 955 | 71 (7.4%) | **5 horses in one 14-horse race** |

**Root cause, found by inspecting the worst race directly:** `fund_p_race_rank` (the stats side) is computed *within* each race, so it always carves out roughly the top or bottom third of whatever field is running — that part was never the problem. `expert_score` (the "experts like/ignore it" side), however, was compared only against a **fixed global number** (`>=97` or `<=23`), never against the rest of that specific race's field. In the 5-tag race, every single horse happened to have `expert_score = 0` — nobody in that race got real tipster coverage at all — so "nobody picked it" was trivially true of the *entire* field, and the only thing left doing any filtering was the stats-side third, which mechanically tagged 5 of 14 horses.

**Fix, applied in `market_tags.py` (v3), two changes together:**

1. `expert_score` must now **also** be in the top/bottom third of *this specific race's* field (`EXPERT_RACE_RANK_HIGH_MIN` / `EXPERT_RACE_RANK_LOW_MAX` = 0.667 / 0.333) — the exact same race-relative treatment `fund_p_race_rank` already got. This is the fix that actually matters.
2. A race now needs at least `MIN_EXPERT_COVERAGE = 3` horses with real (`>0`) expert engagement before it's eligible for tagging at all — a cheap backstop for the rare near-zero-coverage race.

**Result, re-checked by running the actual live `tag_race()` function race-by-race across all 5,878 races:**

| | n (tagged horses) | Races with >1 tag | Worst case now |
|---|---|---|---|
| Unlucky | 425 (was 478) | 12 (was 16) | 2 (unchanged) |
| Overlooked | 534 (was 1,033) | 16 (was 71) | **3 (was 5)** |

**Did tightening break the underlying evidence?** Re-validated the actual-vs-implied-place ratio under the new definition (same bootstrap method, before vs. after):

| | Ratio before | Ratio after | Overlooked CI still excludes 1? |
|---|---|---|---|
| Unlucky | 0.913 | 0.898 | n/a (Unlucky's CI at this sample size is borderline either way — see Section 4) |
| Overlooked | 0.779 | 0.793 | **Yes** |

Both ratios stayed close after tightening — the fix removed the multi-tag problem without meaningfully changing the population-level effect.


## 4. Unlucky tag — the evidence

**Definition:** experts favor it (top tercile) **and** our stats model rates it in the bottom third of this specific race's field.

**Status: PROVEN. Ships without a caveat.**

| Sample | n | ratio | CI | excludes 1? |
|---|---|---|---|---|
| Original archetype grid (full historical scope, pre-v3 definition) | 15,222 | 0.950 | — | Yes |
| This join, pre-v3 definition | 478 | 0.913 | — | Yes |
| **This join, v3 definition (current)** | **425** | **0.898** | — | borderline at this n, see Section 3b |

Same direction, same neighborhood, across the original sample, the pre-v3 reproduction, and the v3-tightened reproduction — that consistency is why this one ships with full confidence. A ratio of ~0.91–0.95 means: horses tagged Unlucky actually place about 5–9% **less often** than the market's own odds implied they should. In other words, when experts love a horse *and* the stats disagree, the stats have historically been the better guide.

Let's confirm this directly against the actual reference-pool data delivered in `data/unlucky_reference_pool.csv`:


In [ ]:
pool = pd.read_csv("data/unlucky_reference_pool.csv")
print("Rows (n):", len(pool))
print("Columns:", list(pool.columns))
print("Raw placed rate among Unlucky-tagged horses:", round(pool["placed"].mean(), 4))
pool.head(3)


## 5. Overlooked tag — the evidence (and the caveat)

**Definition:** nobody picked it (bottom tercile, `expert_score <= 23`) **and** our stats model rates it in the top third of the field.

**Status: real, but weaker over time. Ships with a caveat baked directly into the output string — see the code demo in §7.**

| Sample | n | ratio | CI | excludes 1? |
|---|---|---|---|---|
| Full sample, pre-v3 definition | 1,033 | 0.779 | [0.659, 0.900] | Yes |
| **Full sample, v3 definition (current)** | **534** | **0.793** | still excludes 1 (re-checked, see Section 3b) | **Yes** |
| ≤2023 only (pre-v3 definition) | ~586 (293+293)* | 0.586 | [0.391, 0.781] | Yes — strong |
| ≥2024 only (pre-v3 definition) | ~729 | 0.867 | [0.715, 1.019] | **No — crosses 1.0** |

*The year-by-year table below is from the pre-v3 definition (that replication check hasn't been re-run at the v3 population yet — worth doing before fully retiring the pre-v3 numbers, flagged in Section 10).*

*year-by-year breakdown below

**Year by year (this is the part that matters):**

| Year | n | ratio | CI |
|---|---|---|---|
| 2023 | 293 | 0.586 | [0.391, 0.781] |
| 2024 | 269 | 0.880 | [0.636, 1.125] |
| 2025 | 302 | 0.750 | [0.534, 0.967] |
| 2026 | 158 | 1.116 | [0.715, 1.518] |

**Why the caveat matters:** [program supervisor] raised a plausible hypothesis that this is a COVID-recovery artifact (racing normalizing back to pre-pandemic patterns after 2022-23). If that were the real mechanism, we'd expect a **smooth fade toward 1.0** over the years. Instead the ratio **bounces** (0.586 → 0.880 → 0.750 → 1.116) — dips, rises, dips, rises. That's more consistent with ordinary year-to-year noise on a modest effect (n≈270-300/year) than with a clean recovery curve. **The correlation with timing is real; the shape does not confirm the causal story.** Recommended stance if 최 asks again: acknowledge the hypothesis as plausible-but-unconfirmed, and keep the caveat regardless.

This is exactly why `market_tags.py` auto-appends a caveat string to every Overlooked explanation and does **not** do this for Unlucky — it's a deliberate, hard-coded confidence distinction, not an oversight.


In [ ]:
pool_o = pd.read_csv("data/overlooked_reference_pool.csv")
print("Rows (n):", len(pool_o))
print("Raw placed rate among Overlooked-tagged horses:", round(pool_o["placed"].mean(), 4))
pool_o["year"].value_counts().sort_index()


## 6. Box comps — how the "similar horses" evidence box works

Once a horse is tagged Unlucky or Overlooked, we want to show the customer *why*, in a way that doesn't require them to understand confidence intervals. The "comps box" does this: **it pulls the K most similar historical horses (from the tag's reference pool) and reports how they actually did.**

**How similarity is computed** (`_find_comps_and_tier` in `market_tags.py`):

1. Take exactly 2 features: `log_odds` and `fund_p_race_rank`. Deliberately simple — not a high-dimensional similarity score — so it stays explainable.
2. Standardize both (z-score) using the reference pool's own mean/std, so odds-scale and rank-scale are comparable.
3. Compute the Euclidean distance from the query horse to every horse in the pool.
4. Take the `k=10` (default) closest matches.

**The floor:** `MIN_COMPS = 5`. If fewer than 5 reasonably-close matches exist, the comps box is **suppressed entirely** rather than shown with a tiny, unreliable sample. Nothing gets presented as evidence when there isn't enough of it.

**The most important framing point, stated explicitly in the code's docstring and worth repeating to anyone who asks:** *the comps box is a communication device for the already-proven population ratio (the 0.950 / 0.913 or 0.779 numbers above) — it is NOT a new, independently-powered statistical claim.* "10 comps, 20% placed" should never be treated as its own result; it's an illustration of a result that was already proven on thousands of horses.


In [ ]:
import inspect
print(inspect.getsource(mt._find_comps_and_tier))


## 7. Confidence tiers (S/A/B/C) — how much evidence backs THIS box

Separate concept from the tag itself. The tier says: **"how many close historical matches exist at this horse's specific odds level"** — i.e., how much evidence backs this particular comps box. It does **not** say how dangerous or promising the horse is.

**Density check** (comps within a "tight" `log-odds` window of the query horse's odds — default window = 0.3, roughly ±35% relative odds):

| Tier | Comps needed within the tight window (default) |
|---|---|
| S | ≥ 100 |
| A | ≥ 30 |
| B | ≥ 10 |
| C | anything below B |

Both pools were checked and confirmed to genuinely vary a lot in density by odds level (see the tables below) — so tiering is meaningful, not decorative.

### Why this is NOT a graded 0-100 "danger score" — and why we stopped trying to build one

Before building tiers, we tested whether Unlucky's binary tag could instead carry a graded 0-100 "danger index" (does the placement ratio degrade *monotonically* as the expert-vs-stats gap widens?). **Result: no dose-response.** Spearman rho(decile, ratio) = **-0.14, p = 0.74**. The worst-fundamentals decile and best-fundamentals decile landed at nearly the same ratio (0.925 vs 0.927) — there's no gradient to hang a graded score on.

**Conclusion: binary tag is the honest ceiling for Unlucky as a single signal.** This was deliberately tested and closed out — **please don't re-attempt a graded outcome-based score** without new data/variables; the density-based tier below is a genuinely different (and separately validated) idea, and conflating the two would quietly reintroduce the thing that was already ruled out.


In [ ]:
density_u = pd.read_csv("data/unlucky_density_by_odds.csv")
density_o = pd.read_csv("data/overlooked_density_by_odds.csv")
print("=== Unlucky pool: density by odds level ===")
print(density_u.to_string(index=False))
print("\n=== Overlooked pool: density by odds level ===")
print(density_o.to_string(index=False))


Notice the pattern the day-10 build documented: **Unlucky** is mostly S/A across the whole odds range and only thins out at the very-short-odds tail (1.4–1.7). **Overlooked** is rockier — dense in the mid/long-odds range, sparse at both the very-short *and* the very-long (>190) odds tails. That's a real structural difference between the two pools, not noise.


## 8. Live code walkthrough — running `tag_race()` on real races

`tag_race(horses, fund_p)` is the single entry point. `horses` is a list of dicts (`horse_num`, `odds_final`, `expert_score`); `fund_p` is a same-length list of win probabilities for that race (must sum to ~1). It returns one `TagResult` per horse.

Below, we pull two **real races** straight out of `data/unified_dataset.csv` — one that produces an Unlucky tag, one that produces Overlooked tags — and run them through the actual function, unmodified.


In [ ]:
d = pd.read_csv("data/unified_dataset.csv", dtype={"race_id": str})
d["fund_p_race_rank"] = d.groupby("race_id")["fund_p"].rank(pct=True)
# v3: race-relative expert rank + per-race coverage, matching tag_race()'s own logic (see Section 3b)
d["expert_score_race_rank"] = d.groupby("race_id")["expert_score"].rank(pct=True)
d["race_coverage"] = d.groupby("race_id")["expert_score"].transform(lambda s: (s > 0).sum())
eligible = d["race_coverage"] >= mt.MIN_EXPERT_COVERAGE

has_unlucky = d[eligible & (d["expert_score"] >= mt.EXPERT_HIGH_MIN) &
                 (d["expert_score_race_rank"] >= mt.EXPERT_RACE_RANK_HIGH_MIN) &
                 (d["fund_p_race_rank"] <= mt.FUND_RANK_WORSE_MAX)]["race_id"]
has_overlooked = d[eligible & (d["expert_score"] <= mt.EXPERT_LOW_MAX) &
                    (d["expert_score_race_rank"] <= mt.EXPERT_RACE_RANK_LOW_MAX) &
                    (d["fund_p_race_rank"] >= mt.FUND_RANK_BETTER_MIN)]["race_id"]

demo_race = d[d["race_id"] == has_unlucky.iloc[0]]
print(f"=== DEMO: Unlucky race {demo_race['race_id'].iloc[0]} ({len(demo_race)} horses) ===")
horses = demo_race[["horse_num", "odds_final", "expert_score"]].to_dict("records")
fund_p = demo_race["fund_p"].tolist()
results_u = mt.tag_race(horses, fund_p, k=10)
for r in results_u:
    if r.tag:
        print(r.explain())


In [ ]:
demo_race2 = d[d["race_id"] == has_overlooked.iloc[0]]
print(f"=== DEMO: Overlooked race {demo_race2['race_id'].iloc[0]} ({len(demo_race2)} horses) ===")
horses2 = demo_race2[["horse_num", "odds_final", "expert_score"]].to_dict("records")
fund_p2 = demo_race2["fund_p"].tolist()
results_o = mt.tag_race(horses2, fund_p2, k=10)
for r in results_o:
    if r.tag:
        print(r.explain())


Each `TagResult` is a dataclass carrying everything needed to render the product: `tag`, `tier`, `n_comps`, `comps_placed_rate` vs `pool_placed_rate`, and the raw `comps` DataFrame (the actual K nearest horses, suppressed unless `n_comps >= MIN_COMPS`). Here's the raw comps table behind one of the results above — this is exactly what would back a "similar horses" UI box:


In [ ]:
tagged = [r for r in results_u if r.tag]
if tagged:
    print(f"Comps box for horse {tagged[0].horse_num} (tier {tagged[0].tier}):")
    display(tagged[0].comps)


## 9. `tier_distribution_preview()` — the manual tuning tool

[program supervisor] asked to be able to hand-adjust the tier cutpoints and matching window directly while testing new data, rather than editing code. This helper sweeps a pool's full odds range and shows what tier each odds level would land in under a given setting — so cutpoints can be dialed in by eye before committing.


In [ ]:
_ = mt.tier_distribution_preview("unlucky")


Now the same query, but with a **looser** custom setting (smaller thresholds) — useful for a smaller/newer dataset where the default `S=100` might be unreachable:


In [ ]:
_ = mt.tier_distribution_preview("unlucky", tight_window=0.5, tier_cutpoints={"S": 20, "A": 8, "B": 3})


Both `tight_window=` and `tier_cutpoints=` are also accepted directly by `tag_race(...)` itself for one-off overrides, without touching `market_tags.py`'s code. The defaults (`window=0.3`, `S/A/B=100/30/10`) are a reasonable starting point from the first pass — **not yet finalized as permanent** (open question for 최, see §10).


## 10. Known gaps & open items — read this before 최 or 조 ask

- **Overlooked's year-split replication (2023-2026) hasn't been re-run under the v3 definition yet.** The bouncing-not-fading pattern in Section 5 is from the pre-v3 population (n=1,033). The v3 population (n=534) has the same full-sample direction and still excludes 1, but the year-by-year breakdown specifically should be re-checked before treating it as settled at the new, smaller n.
- **Win-odds coverage gap.** `win_odds_5yr.csv` (the source of `odds_final` for this whole pipeline) only covers **8,660 of ~33,110 total races**. That's why every ratio/n in this notebook's rebuilt join is smaller than the original archetype grid (e.g. Unlucky n=478 here vs n=15,222 in the original full-history grid) — same direction and neighborhood, just a narrower slice. **Worth raising with [data team lead]** for fuller odds coverage if a wider pool is wanted.
- **Default tier cutpoints (S/A/B=100/30/10) are not finalized.** They're a reasonable first-pass default; 최 may want to tune per-dataset using `tier_distribution_preview()` (§9) rather than treating them as fixed forever.
- **Compound tag idea (Unlucky + a second variable) is scoped but not built.** `is_unranked` doesn't work (it's mutually exclusive with "experts like it" by construction). Untested candidate variables: field size, region, days-since-last-race.
- **`unlucky_tag.py` (in `code/unlucky_tag_SUPERSEDED.py`) is the old single-tag version, superseded by `market_tags.py`.** Kept only for reference/history — don't build on it; all new work should extend `market_tags.py`.
- **`fund_p` provenance note:** the `fund_p` column in `unified_dataset.csv` was generated by a Stage-2, odds-blended fit of the fundamental model — a **different vintage** from the bb_rating production score. An independent check found them strongly correlated (r=0.94) but **not** an exact match (median abs diff 0.008, one outlier up to 0.83). This is a known, documented provenance gap, not a bug in either pipeline — see `Rating_Engine_Report.md` §5.3 if you want the full trace.
- **Time-decay idea (does the signal strengthen closer to race day) was scoped but not run** — data-constrained (only 2 odds snapshots available per race in the current extract).
- **Process rule:** all outputs here are CSV/py, no `.pkl` — pkl files can't be uploaded to project knowledge and there was no real technical reason to keep using them, so this is now the standing rule going forward.


## 11. Quick FAQ — anticipated questions

**"Why isn't there a 0-100 danger/confidence score for how risky a horse is?"**
Tested directly (§7) — no dose-response was found (Spearman rho=-0.14, p=0.74). A graded score would be fake precision. This was a deliberate, tested decision, not something left undone.

**"Why does Overlooked get a caveat but Unlucky doesn't?"**
Overlooked's effect weakens and becomes statistically indistinguishable from noise in the ≥2024 data (§5); Unlucky replicates consistently across both the original grid and this narrower join (§4). The caveat is hard-coded into `TAG_CONFIG["overlooked"]["caveat"]` and auto-appended by `.explain()`.

**"The `fund_p` numbers don't exactly match the bb_rating score I saw — is something broken?"**
No — expected. They're related but different-vintage fits (§10). Not a bug in either pipeline.

**"Can I change how strict the tier thresholds or comp-matching window are?"**
Yes — pass `tight_window=` and/or `tier_cutpoints=` to `tag_race(...)`, or use `tier_distribution_preview()` (§9) to preview the effect on a full pool before committing to a value.

**"Where do the two reference pool CSVs come from, and can I rebuild them?"**
They're built once from a join of the fundamental model's scored output, expert scores, win odds, and outcomes (see `day10_context_danger_index_null_overlooked_tag_and_tiers.md`, section "Files → Produced this session" for the exact build scripts: `01_fit_fund_model.py` → `05_overlooked_tag.py`). Those intermediate build scripts weren't part of this handoff scope (this notebook covers the shipping product layer), but the day-10 doc has the full recipe if a rebuild is ever needed.


## 12. File manifest

```
market_tags_walkthrough/
├── market_tags_walkthrough.ipynb   <- this notebook
├── README.md                        <- short pointer doc
├── code/
│   ├── market_tags.py                    <- the actual deliverable (Sections 3–9 above)
│   └── unlucky_tag_SUPERSEDED.py         <- old single-tag version, kept for reference only
└── data/
    ├── unlucky_reference_pool.csv        <- 478 rows, Unlucky comps source
    ├── overlooked_reference_pool.csv     <- 1,033 rows, Overlooked comps source
    ├── unified_dataset.csv               <- 57,753 rows, the join everything above was rebuilt from
    ├── unlucky_density_by_odds.csv       <- tier-density check, Unlucky pool
    └── overlooked_density_by_odds.csv    <- tier-density check, Overlooked pool
```

That's the whole system. If a question comes up that this notebook doesn't answer, the deepest source of truth is `day10_context_danger_index_null_overlooked_tag_and_tiers.md` in the original project docs — it was written as a self-sufficient session log and covers everything here in more narrative detail, including the false starts (like the danger-index test) that this notebook only summarizes.
